In [3]:
# React patter 
## LLM first think then act, then think again, and finally act again.
import re
import httpx
import os
import requests 
import json
from dotenv import load_dotenv

_ = load_dotenv()

In [ ]:
## test on local llm
response = requests.post(
    "http://localhost:1234/api/v1/chat", 
    headers = {
        "Content-Type": "application/json"
    }, 
    json = {
        "model": "nvidia/nemotron-3-nano-4b", 
        "input": "hello world"
    }
)

print(json.dumps(response.json(), indent=2))

{
  "model_instance_id": "nvidia/nemotron-3-nano-4b",
  "output": [
    {
      "type": "reasoning",
      "content": "The user says \"hello world\". Likely they want a greeting. Should respond friendly.\n"
    },
    {
      "type": "message",
      "content": "\n\nHello! How can I assist you today? \ud83d\ude0a"
    }
  ],
  "stats": {
    "input_tokens": 19,
    "total_output_tokens": 33,
    "reasoning_output_tokens": 18,
    "tokens_per_second": 9.255202966601058,
    "time_to_first_token_seconds": 0.429
  },
  "response_id": "resp_2898f74f2d308d79d909815bb9e0a5055d99141c7bb5d167"
}


In [11]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})
    
    ## whats the agent going to do
    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result
    ## exectuee the function
    def execute(self):
        completion = requests.post(
            "http://localhost:1234/api/v1/chat",
            headers={"Content-Type": "application/json"},
            json={
                "model": "nvidia/nemotron-3-nano-4b",
                "input": self.messages[-1]["content"]
            }
        )
        return json.dumps(completion.json(), indent=2)
    

In [12]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [13]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [14]:
abot = Agent(prompt) ## initilize agent


In [15]:
result = abot("how much does a golden retriever weigh?")

In [16]:
print(result)

{
  "model_instance_id": "nvidia/nemotron-3-nano-4b",
  "output": [
    {
      "type": "reasoning",
      "content": "Okay, the user is asking about the weight of a Golden Retriever. Let me start by recalling the typical weight range for this breed. I remember they're medium to large dogs, so it's not a fixed number. The user might be considering getting one or already has one and wants to check their health.\n\nFirst, I should confirm the standard weights. Males usually weigh between 50-65 pounds, females 45-60. But wait, those are in pounds. Maybe I should also mention kilograms since some users prefer metric. The user didn't specify units, so including both would be helpful.\n\nHmm, why does the user need this info? If they're a potential owner, they might want to know if their puppy will reach that weight by adulthood. Or maybe they're concerned about their dog's health\u2014like obesity or underweight. Golden Retrievers are prone to weight issues, so emphasizing monitoring is imp

In [17]:
result = average_dog_weight("Golden Retriever")

In [18]:
result

'An average dog weights 50 lbs'

In [19]:
next_prompt = "Observation: {}".format(result)

In [20]:
abot(next_prompt)

'{\n  "model_instance_id": "nvidia/nemotron-3-nano-4b",\n  "output": [\n    {\n      "type": "reasoning",\n      "content": "Okay, the user mentioned that an average dog weighs 50 lbs. Hmm, I need to check if that\'s accurate. Let me think about common dog breeds and their typical weights.\\n\\nFirst, I recall that dogs come in various sizes. Small breeds like Chihuahuas are usually under 10 pounds, while large ones like Great Danes can be over 100 pounds. The average weight across all breeds... Wait, maybe the user is thinking of a specific breed? But they said \\"average dog,\\" which might imply overall.\\n\\nI remember that many sources cite an average adult dog weighing around 50-60 lbs. Let me verify that. A quick mental check: AKC data shows the median weight for all breeds is about 51 pounds. Yeah, that\'s right. So 50 lbs is a common average figure used in general contexts.\\n\\nBut wait, is the user aware that this is an average? Maybe they\'re concerned because their dog wei

In [ ]:
## structure for the manual react agent: 
## Thought: I should look the dogs weight using average_dog_weight
## Action: average_dog_weight: Bulldog


In [ ]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

In [ ]:
# loop for the agent to think, act and observe
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return